# 2.1 BasePlot 101: instellingen, lagen en multiplots

Dit notebook is kort en bevat geen ingewikkelde onderzoeksconclusies die je hoeft te verdedigen — net zoals [01.1-goad-toolkit-101](../lesson1/01.1-goad-toolkit-101.ipynb) dat deed voor `TransformBase`. In `02.2-comparing_categories.ipynb` ga je wat je hier leert toepassen op echte data; dit notebook is er puur voor jou, `BasePlot`, en een piepklein speelgoeddatasetje dat je makkelijk kunt overzien.

Als overerving (*subclassing*) en klassen nog nieuw voor je zijn, lees dan eerst [00.1-classes](../lesson0/00.1-classes.ipynb) — `BasePlot` pakt de draad precies op waar dat notebook eindigt, op dezelfde manier als `TransformBase` dat deed.

In [ ]:
# Paragraaf 2.1: Benodigde bibliotheken importeren
import matplotlib.pyplot as plt  # De standaard tekenbibliotheek van Python
import pandas as pd              # Voor het werken met tabellen (DataFrames)
import seaborn as sns            # Handige bibliotheek voor statistische grafieken (zoals staafdiagrammen)
# Hier importeren we de specifieke bouwstenen van docent Raoul:
from goad_toolkit.visualizer import (
    BasePlot,           # De 'basis-tekenaar' (de blauwdruk voor alle grafieken)
    GroupedBarPlot,     # Een kant-en-klaar gegroepeerd staafdiagram
    HighlightCategory,  # Een 'laag' om 1 categorie een accentkleur te geven
    PlotSettings,       # De 'stijlgids' (afmetingen, titels, raster, etc.)
)
from notebooktester import param # Hulptooltje voor automatische controles in het notebook
# We maken een heel simpel test-tabelletje met 8 rijen om mee te oefenen:
# 4 regio's (north, south, east, west) over 2 kwartalen (Q1, Q2) met hun omzet (revenue)
sales = pd.DataFrame({
    "region": ["north", "south", "east", "west", "north", "south", "east", "west"],
    "quarter": ["Q1", "Q1", "Q1", "Q1", "Q2", "Q2", "Q2", "Q2"],
    "revenue": [120, 95, 110, 130, 140, 100, 115, 125],
})
sales  # Toon de tabel op het scherm

## 2.1.1 Het probleem dat BasePlot oplost

Een grafiek in een Jupyter Notebook begint meestal als één onschuldige aanroep van `seaborn`, maar eindigt in elke cel als een wirwar van opmaakregels:

In [ ]:
# Paragraaf 2.1.1: De klassieke manier (5 regels opmaak per cel)
fig, ax = plt.subplots(figsize=(8, 4))                  # 1. Maak handmatig een leeg canvas van 8 bij 4 inch
sns.barplot(data=sales, x="region", y="revenue", ax=ax)  # 2. Teken de staven met seaborn
ax.set_title("Revenue")                                 # 3. Stel de titel handmatig in
ax.set_ylabel("revenue (k$)")                           # 4. Stel het label van de y-as handmatig in
ax.grid(True, alpha=0.3)                                # 5. Zet handmatig een lichtgrijs raster aan
# Het probleem: als je dit in 10 cellen doet, vergeet je snel iets en wordt de styling rommelig!

Als je samenwerkt met AI (LLM's) om code te schrijven, eindig je al snel met een totale chaos aan stijlen. Je krijgt weliswaar grafieken op je scherm, maar het wordt ontzettend lastig om een consistente en beheerbare codebase te behouden.

Eén manier om dit strak te organiseren is door te werken met abstracte basisklassen (`BasePlot`) en losse instellingen (`PlotSettings`).

De onderstaande code helpt je te focussen op de hoofdzaak: we maken een staafdiagram (*bar plot*), en we hebben herbruikbare instellingen voor de stijl en opmaak.

Dit maakt het veel eenvoudiger om instellingen te hergebruiken en het programmeerproces op een hoger, abstracter niveau aan te sturen.

In [ ]:
# Paragraaf 2.1.1: Oplossing met BasePlot en PlotSettings
# 1. DE TEKENAAR (BarPlot erft alle basisfunctionaliteit over van BasePlot)
class BarPlot(BasePlot):
    """Een staafdiagram. Alle styling wordt geregeld door PlotSettings."""
    def build(self, data: pd.DataFrame, x: str, y: str, **kwargs):
        # self.ax is al klaargezet door BasePlot met de juiste afmetingen en labels!
        sns.barplot(data=data, x=x, y=y, ax=self.ax, **kwargs)  # Teken alleen de staven
        return self.fig, self.ax  # Verplicht: return altijd fig en ax zodat andere lagen hierop door kunnen bouwen!
# 2. DE HUISSTIJL / STIJLGIDS (Herbruikbare instellingen)
settings = PlotSettings(
    figsize=(8, 4),           # Afmetingen van de grafiek (breedte x hoogte)
    title="Revenue",          # Hoofdtitel van de grafiek
    ylabel="revenue (k$)",     # Y-as label (expliciet vermelden dat het om k$ gaat)
    grid=True,                # Zet rasterlijnen aan
    grid_alpha=0.3,           # Maak het raster subtiel (30% dekking)
    # Let op: xlabel is niet ingevuld (staat op None), dus die pakt automatisch de kolomnaam!
)

In [ ]:
fig, ax = BarPlot(settings).plot(data=sales, x="region", y="revenue")

Zie je hoe `settings` precies zoals hij is hergebruikt wordt voor een compleet andere groepering? Geen `dataclasses.replace`, geen nieuw object bouwen, helemaal niets. 

Dat is geen toeval: `settings` hierboven had `xlabel` nooit ingesteld, dus die staat standaard op `None`. Een label van `None` betekent dat `create_figure` de as met rust laat en de kolom die je plotte (`x=`) zijn eigen naam laat leveren: de eerste keer `"region"`, de tweede keer `"quarter"`. Alleen voor de Y-as (`ylabel`) was een expliciete overschrijving nodig, omdat `"revenue (k$)"` veel duidelijker is voor een lezer dan de kale databasenaam `"revenue"`. 

De regel in één zin samengevat: **laat een as-label leeg (`None`) als de kolomnaam al perfect is als label; stel het alleen in wanneer de kolomnaam en het label dat een lezer verwacht wezenlijk van elkaar verschillen.**

In [ ]:
# BarPlot(settings) koppelt de tekenaar aan de huisstijl
# .plot(...) maakt het canvas aan, zet titels/assen klaar, en tekent de staven
fig, ax = BarPlot(settings).plot(data=sales, x="region", y="revenue")

Een label niet instellen helpt natuurlijk alleen als de kolomnaam zélf al het gewenste label is. Soms is dat niet zo, of wil je de rest van `settings` onaangeroerd laten maar één veld aanpassen — bijvoorbeeld een specifieke titel voor een grafiek die verder qua stijl identiek is. 

Geef dat veld dan direct mee aan de aanroep van `.plot()`: elk argument dat overeenkomt met een veld in `PlotSettings` overschrijft dat veld **alleen voor deze ene aanroep**, terwijl het gedeelde `settings`-object onaangetast blijft. 

Dat is de taakverdeling van dit ontwerp in één regel: **de styling leeft in de instellingen (`PlotSettings`), en de inhoudelijke claim (een titel *is* een claim) leeft bij het aanroepen van de plot, direct naast de datakoppeling waar hij bij hoort.**

In [ ]:
# We filteren alleen op regio 'west', en geven alleen deze ene grafiek een eigen titel:
fig, ax = BarPlot(settings).plot(
    data=sales[sales.region == "west"], 
    x="quarter", 
    y="revenue",
    title="West region, on its own",  # <-- Alleen deze specifieke plot krijgt deze titel
)

Hetzelfde plaatje, en "maak elke figuur breder" is nu één kleine aanpassing in één `PlotSettings`, in plaats van vijftien aanpassingen in vijftien losse cellen.

**Wat `.plot()` je garandeert:** Hij voert automatisch `create_figure()` uit *voordat* hij jouw eigen `build()` aanroept. Tegen de tijd dat jouw code start, heb je dus al:
- `self.fig` en `self.ax`, direct aangemaakt met het juiste formaat (`settings.figsize`) — je hoeft zelf dus nooit meer `plt.subplots` aan te roepen;
- de titel, as-labels en het raster (*grid*) zijn al netjes toegepast vanuit de instellingen;
- bij `n_plots > 1` (paragraaf 2.1.2 hieronder) staat er al een grid van deelgrafieken klaar, waarbij ongebruikte assen netjes verborgen zijn.

Jouw `build()`-methode heeft daardoor nog maar **één taak**: tekenen op `self.ax`. En één verplichting: **retourneer altijd `(self.fig, self.ax)`**, omdat andere functies en lagen (`plot_on`) daarop doorkoppelen.

### Een waarschuwing: automatische labels tonen de kolomnaam, geen mooie toelichting

`None` betekent niet automatisch "mooi" — het betekent letterlijk "wat de kolom ook maar heet". Automatische naamgeving is een prima standaardkeuze wanneer de kolomnaam al leest als een nette titel (`region`, `quarter`). Het is echter een slechte keuze zodra dat niet het geval is — en bij echte datasets is dat bijna altijd zo.

In [ ]:
# Stel: in je brondata heet de kolom 'revenue_usd_k' (een lelijke database-naam)
ugly = sales.rename(columns={"revenue": "revenue_usd_k"})
# Omdat we hier geen ylabel hebben opgegeven in PlotSettings, pakt hij letterlijk 'revenue_usd_k'
fig, ax = BarPlot(PlotSettings(figsize=(8, 4), title="Auto-label shows exactly the column name")).plot(
    data=ugly, x="region", y="revenue_usd_k"
)
print(f"ylabel: {ax.get_ylabel()!r}")  # Toont: 'revenue_usd_k' (dit moet je dus altijd zelf netjes maken!)

**Een Y-as met `ylabel: 'revenue_usd_k'` is geen foutje van Seaborn; het is een `PlotSettings` die je bent vergeten in te vullen.** 

Automatische labeling bespaart je het intypen van een label wanneer de kolomnaam al duidelijk genoeg is — het was nooit bedoeld om je te ontslaan van de plicht om na te denken of dat label wel begrijpelijk is voor je lezer.

## 2.1.2 Multiplot (Meerdere panelen naast elkaar)

`create_figure` bepaalt de layout al vóórdat jouw `build()` ooit start:

- `n_plots=None` (of `1`): één enkele as, met het formaat zoals opgegeven in `settings.figsize`.
- `n_plots > 1`: een raster (*grid*), met standaard maximaal `min(settings.max_cols, n_plots)` kolommen breed (standaard 3 kolommen), dat automatisch doorloopt naar zoveel rijen als nodig is. Ongebruikte assen worden automatisch verborgen in plaats van leeg en verwarrend achter te blijven. Elke zichtbare as krijgt de bijbehorende titel uit `settings.subplot_titles`, en de totale figuur krijgt `settings.title` als hoofdtitel (`suptitle`) erboven.

`plot_on_axes(other_plot, ax, ...)` is de manier waarop je een *specifieke* plot-instantie op één as van dat raster plaatst — het meer-panelen equivalent van `.plot()`.

In [ ]:
# Paragraaf 2.1.2: Twee grafieken naast elkaar (multiplot)
pair = PlotSettings(
    figsize=(11, 4),                                                        # Breder canvas voor 2 grafieken
    title="One question, two categorical variables",                         # Overkoepelende titel bovenaan
    subplot_titles=["revenue by region", "revenue by region and quarter"], # Titels per paneel
    xlabel="region",
    ylabel="revenue (k$)",
)
host = BarPlot(pair)                      # Maak de hoofdplot met deze instellingen
fig, axes = host.create_figure(n_plots=2) # Maak 2 panelen aan; axes[0] is links, axes[1] is rechts
# Teken plot 1 op het linkerpaneel (axes[0]):
host.plot_on_axes(BarPlot(pair), axes[0], data=sales, x="region", y="revenue", color="#cccccc")
# Teken plot 2 op het rechterpaneel (axes[1]) als gegroepeerd staafdiagram:
host.plot_on_axes(GroupedBarPlot(pair), axes[1], data=sales, x="region", y="revenue", hue="quarter")

### Een waarschuwing over `max_cols`

Het aantal kolommen in het raster is `min(settings.max_cols, n_plots)`, en `max_cols` staat standaard op drie. Vier panelen met die standaardwaarde worden dus gewrapt naar een bovenste rij van drie panelen en een eenzaam vierde paneel op de tweede rij — zelden wat je wilt bij een set van vier. Stel daarom `max_cols=2` in voor een net 2x2 vierkant raster, of `max_cols=4` voor één lange rij van vier panelen naast elkaar.

In [ ]:
# Paragraaf 2.1.2: Vier small multiples (één paneel per regio)
quad = PlotSettings(
    figsize=(9, 6),                              # Hoogte en breedte voor een 2x2 raster
    title="One panel per region",                 # Overkoepelende titel
    subplot_titles=sorted(sales.region.unique()),  # De 4 regio-namen als titels per paneel
    xlabel="quarter",
    ylabel="revenue (k$)",
    max_cols=2,                                   # Dwing af: 2 kolommen breed (zodat het een net 2x2 vierkant wordt!)
)
host = BarPlot(quad)
fig, axes = host.create_figure(n_plots=4)        # Maak 4 panelen aan
# Loop door elk paneel (ax) en filter de data voor die specifieke regio:
for ax, region in zip(axes, sorted(sales.region.unique())):
    host.plot_on_axes(
        BarPlot(quad), 
        ax,
        data=sales[sales.region == region], 
        x="quarter", 
        y="revenue",
        color="#cccccc"                          # Eerst neutraal grijs volgens ontwerpprincipes
    )

Dat is een *small-multiples* figuur (kleine deelgrafieken naast elkaar), en het kostte je slechts één extra veld in `PlotSettings`. 

Voorbij de vijf overlappende lijnen of een tiental staven kan niemand meer zinnig iets vergelijken op één enkele as. Eén paneel per groep, met een gedeelde Y-as, is dan vrijwel altijd de beste oplossing — en `create_figure(n_plots=n)` maakt het zo eenvoudig dat je dit ook daadwerkelijk snel gebruikt. 

Dit patroon komt zó vaak voor dat `goad` het meelevert als een kant-en-klare klasse: `FacetPlot` tekent elke willekeurige plot één keer per categorie in een enkele aanroep (met `sharey=True` in de instellingen voor die gedeelde Y-as) — in Les 3 ga je dit gebruiken op echte data!

## 2.1.3 Lagen stapelen: `plot_on` en `plot_on_axes`

Twee verwante methoden met twee verschillende taken:

- **`plot_on(other_plot, ...)`**: Geeft een andere plot-instantie *dezelfde as* en roept diens `plot` aan. Beide visualisaties belanden dus op dezelfde set assen. Dit is dé manier waarop je een markering (*highlight*), een trendlijn of een tweede datareeks toevoegt aan een bestaande grafiek.
- **`plot_on_axes(other_plot, ax, ...)`**: Geeft een andere plot-instantie *een specifieke benoemde as binnen een figuur met meerdere panelen* — het mechanisme dat we in 2.1.2 zojuist twee keer hebben gebruikt.

`HighlightCategory` is de richtlijn *"eerst grijs, dan één kleur"* uitgewerkt als een `plot_on`-laag. Hij kleurt simpelweg de staven opnieuw die al op de as staan. Daardoor werkt hij over een `BarPlot`, over een kale `sns.barplot`, of over wat dan ook dat staven heeft getekend — hij hoeft niet eens te weten hoe die staven er oorspronkelijk op zijn gekomen.

In [ ]:
# Paragraaf 2.1.3: Lagen stapelen met plot_on
highlight = "west"  # We willen regio 'west' laten opvallen
pointed = PlotSettings(
    figsize=(8, 4),
    title=f"{highlight.title()} is the strongest region",  # Storytelling titel die direct de conclusie vertelt
    xlabel="region",
    ylabel="revenue (k$)",
    highlight=[highlight],                                # Vertel de stijlgids welke categorie moet opvallen
)
bars = BarPlot(pointed)
# Stap 1: Teken eerst alle staven neutraal grijs
bars.plot(data=sales, x="region", y="revenue", color="#cccccc")
# Stap 2: Leg met plot_on de 'HighlightCategory'-laag eroverheen (kleurt 'west' automatisch in!)
bars.plot_on(HighlightCategory(pointed))

### Twee regels om code herbruikbaar en stapelbaar te houden

1. **Teken altijd op `self.ax` en `self.fig`, nooit op `plt`.** Aanroepen op moduleniveau zoals `plt.xticks(...)` werken altijd op de op dat moment *actieve* as van matplotlib. In een figuur met meerdere panelen is dat meestal niet de as die jouw specifieke plot heeft gekregen! `self.ax.tick_params(axis="x", rotation=45)` is de expliciete vorm, en dat is de enige manier waarop `plot_on_axes` betrouwbaar kan werken.
2. **`build()` retourneert ALTIJD `(self.fig, self.ax)` — ook bij lagen.** Zowel `plot_on` als `plot_on_axes` schakelen door op die retourwaarde. Een `build()` die iets anders retourneert, breekt de keten voor degene die de laag daarna aanroept.

## 2.1.4 Waarom roepen we niet gewoon direct `sns.barplot(...)` aan?

Voor de eenvoudige grafiek hierboven kun je je eerlijk afvragen: waarom al die moeite? Eén `sns.barplot` aanroep en een paar `ax.set_*` regels leveren je hetzelfde plaatje op in minder regels code dan de klasse-versie. Voor een eenmalige grafiek die je daarna nooit meer aanraakt, is die overhead reëel. `BasePlot` is dan ook geen dwingende wet voor elk los grafiekje.

De klasse verdient zichzelf echter dubbel en dwars terug zodra een grafiek méér wordt dan één simpele Seaborn-aanroep — en dat gebeurt sneller dan je denkt. `ComparePlotDate`, die al standaard in `goad_toolkit.visualizer` zit, bestaat uit slechts twee regels en tweemaal een `plot_on`:

```python
class ComparePlotDate(BasePlot):
    def build(self, data, x, y1, y2, date, datelabel, **kwargs):
        compare = LinePlot(self.settings)
        self.plot_on(compare, data=data, x=x, y=y1, label=y1, **kwargs)
        self.plot_on(compare, data=data, x=x, y=y2, label=y2, **kwargs)
        self.plot_on(VerticalDate(self.settings), date=date, label=datelabel)
        ...
```

Dit is geen schets, maar echte werkende code. Draai de onderstaande cel maar eens: twee omzetkanalen, zes maanden, en een beleidswijziging die laat zien wie er wint:

In [ ]:
# Paragraaf 2.1.4: Waarom BasePlot handig is bij meerdere lijnen en datums
from goad_toolkit.visualizer import ComparePlotDate
# Maak een tijdreeks-dataset: 6 maanden omzet (online vs fysieke winkel)
timeline = pd.DataFrame({
    "month": pd.date_range("2024-01-01", periods=6, freq="MS"),
    "online": [80, 85, 90, 130, 140, 150],
    "in_store": [150, 145, 140, 100, 95, 90],
})
compare_settings = PlotSettings(
    figsize=(9, 4),
    title="Online overtakes in-store after free shipping launches",
    ylabel="revenue (k$)",
)
# ComparePlotDate tekent automatisch twee lijnen én een verticale datumlijn in 1 aanroep:
fig, ax = ComparePlotDate(compare_settings).plot(
    data=timeline, 
    x="month", 
    y1="online", 
    y2="in_store",
    date="2024-04-01", 
    datelabel="free shipping launches",  # Annotatietekst bij de datumlijn
)

Drie methode-aanroepen — `plot`, en daarna twee impliciete `plot_on`'s die je zelf niet hoefde te schrijven — produceerden twee gelabelde lijnen, een legenda en een geannoteerde datumlijn. En `ComparePlotDate` zelf is puur opgebouwd uit compositie: geen eigen `sns`-aanroep, geen as-instellingen, alleen `LinePlot` die twee keer wordt aangeroepen met `VerticalDate` er overheen gelegd. 

Dát is de echte winst van `BasePlot`: niet dit ene grafiekje, maar het feit dat *deze* complexe grafiek vrijwel niets kostte om te schrijven omdat `LinePlot` en `VerticalDate` al bestonden.

Twee lijnen die je vandaag tekent (een markering, een vergelijking) worden morgen drie of vier composities, elk met een `plot_on`-aanroep in plaats van een herschreven Seaborn-tovertruc. `DecomposePlot` overschrijft `plot` volledig omdat hij zijn aantal panelen (vier, vast) moet weten vóórdat de figuur bestaat; `PlotFits` loopt over een willekeurig aantal gefitte verdelingen en kleurt elk paneel op basis van wie er won. Niets daarvan is uit te drukken als "één simpele Seaborn-aanroep" — het is uit te drukken als kleine, gestapelde plots. En dát is precies waar `BasePlot` voor bedoeld is.

Wil je hier meer over lezen? Met `goad_list_concepts` en `goad_get_concept("03-plot-composition")` (de ingebouwde coach van deze cursus) loop je stap voor stap door de compositie-voorbeelden met de bijbehorende argumentatie. Ook de broncode van de toolkit zelf — `goad_toolkit/visualizer.py` — is zeker het bekijken waard zodra je zelf een subklasse hebt geschreven.

## 2.1.5 Waarom dit zo waardevol is als je met AI (LLM's) werkt

Laten we eerlijk zijn over de praktijk: je vraagt regelmatig een AI-assistent om code voor je grafieken te schrijven. Dat maakt het nut van `BasePlot` niet kleiner, maar juist veel groter!

Als je een AI simpelweg vraagt: *"plot omzet per regio"*, zonder verdere context, verzint het model elke keer opnieuw een eigen stijl — hier een ander kleurenpalet, daar een grid, soms wel as-labels en soms niet, de ene keer `figsize=(10, 6)` en de volgende keer `(8, 5)` zonder enige logica. Vraag je om diezelfde grafiek in een notebook waar al vijftien ad-hoc Seaborn-cellen staan, dan krijg je een zestiende stijl, simpelweg omdat er niets was om consistent mee te zijn.

Een contract van `PlotSettings` + `BasePlot` geeft de AI een vaste kapstok. De instructie *"Schrijf een `BasePlot`-subklasse met deze `build`-signatuur, opgemaakt via deze `PlotSettings`"* is een heldere specificatie waar een LLM zich exact aan kan houden — net zoals een typehint een afspraak is die gecontroleerd kan worden. 

Bovendien kan het resultaat direct samenwerken met `plot_on` en `plot_on_axes`, in plaats van dat het een eenzaam stukje code is dat niemand anders kan hergebruiken. Dit is cruciaal voor pipelines die volgende maand op nieuwe data moeten blijven werken, zonder dat jij elke cel opnieuw hoeft te ontcijferen.

## 2.1.6 Jouw beurt (Oefening)

Schrijf een `BasePlot` **laag** — een subklasse die zelf niets nieuws tekent, maar alleen de bestaande staven op de as opnieuw kleurt. 

`HighlightCategory` hierboven is het voorbeeld: zoek de hoogste staaf (`patch.get_height()`) en geef die een opvallende kleur (`facecolor`), terwijl je de rest neutraal grijs laat.

```python
class MaxHighlight(BasePlot):
    def build(self, color: str = "crimson", **kwargs):
        # Vind de hoogste patch op self.ax, kleur die opnieuw; laat de rest grijs
        ...
        return self.fig, self.ax
```

Pas dit toe met `bars2.plot_on(MaxHighlight(settings))` in de cel hieronder, op exact dezelfde manier als `HighlightCategory` eerder werd toegepast.

In [ ]:
# Paragraaf 2.1.6: Jouw beurt — Automatisch de hoogste staaf een kleur geven

# 1. Definieer de MaxHighlight laag:
class MaxHighlight(BasePlot):
    """Een laag die de hoogste staaf opzoekt en herkleurt."""
    def build(self, color: str = "crimson", **kwargs):

        # Zoek alle staven (patches) op de as die niet leeg zijn:
        patches = [p for p in self.ax.patches if p.get_width() != 0]
        if patches:
            # Vind de staaf met de maximale hoogte (hoogste omzet):
            tallest = max(patches, key=lambda p: p.get_height())
            # Geef alleen die hoogste staaf de accentkleur (bijv. crimson rood):
            tallest.set_facecolor(color)
        return self.fig, self.ax  # Return altijd fig en ax zodat de keten intact blijft!

# 2. Teken eerst de staven neutraal grijs:
bars2 = BarPlot(settings)
bars2.plot(data=sales, x="region", y="revenue", color="#cccccc")

# 3. Pas de MaxHighlight laag toe met plot_on (kleurt de hoogste staaf rood):
bars2.plot_on(MaxHighlight(settings))

# Automatische test van het notebook (controleert of er minimaal 2 verschillende kleuren zijn):
MIN_DISTINCT_COLORS = param(2, test=1)
colors = {tuple(patch.get_facecolor()) for patch in bars2.ax.patches if patch.get_width() != 0}
assert len(colors) >= MIN_DISTINCT_COLORS, (
    "define a MaxHighlight layer above and apply it with bars2.plot_on(...), so the bars "
    "are not all the same colour"
)
print(f"{len(colors)} distinct bar colour(s)")  # Toont: 2 distinct bar colour(s) -> TEST GESLAAGD!

---

**Hoe we nu verder gaan:** In `02.2-comparing_categories.ipynb` leiden we opnieuw een `BarPlot` af — ditmaal op échte data (pinguïns en de Titanic) — en gebruiken we `GroupedBarPlot`, `HeatmapPlot` en `HighlightCategory` om de vraag te beantwoorden waar deze les werkelijk over gaat: **hoe verschillen groepen van elkaar, en op welke manieren kan die vergelijking gruwelijk misleidend zijn?**